# 03c: Statistical Model Comparison

**Purpose:** Rigorous statistical comparison of all baseline models

**Dataset:** COMPAS predictions from all baseline models

**Date:** 2025-11-08

---

## Overview

### Purpose
Statistically compare baseline models using:
- **DeLong test**: Compare AUROC values
- **McNemar's test**: Compare prediction agreement
- **Permutation tests**: Compare arbitrary metrics
- **Multiple comparison corrections**: Bonferroni, Holm, Benjamini-Hochberg
- **Effect sizes**: Practical significance beyond p-values

### Models Compared
1. Logistic Regression (baseline)
2. XGBoost
3. LightGBM
4. CatBoost

### Statistical Framework
- **Null hypothesis**: No difference between models
- **Significance level**: α = 0.05 (with corrections)
- **Effect size threshold**: Cohen's d ≥ 0.2 (small effect)

### Outputs
- Statistical test results → `results/metrics/model_comparison_stats.json`
- Comparison table → `results/tables/baseline_model_comparison.csv`
- Visualizations → `results/figures/model_performance/`

### Runtime: 2-5 minutes

---

In [ ]:
# Setup
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, f1_score
)
from scipy import stats

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / "src"))

# Our statistical utilities
from statistics.hypothesis_tests import mcnemar_test, delong_test, permutation_test
from statistics.effect_sizes import cohens_d, number_needed_to_evaluate
from statistics.multiple_comparisons import holm_correction, benjamini_hochberg

# Directories
PROCESSED_DIR = project_root / "data" / "processed"
PREDICTIONS_DIR = project_root / "results" / "predictions"
METRICS_DIR = project_root / "results" / "metrics"
TABLES_DIR = project_root / "results" / "tables"
FIGURES_DIR = project_root / "results" / "figures" / "model_performance"

for d in [TABLES_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
ALPHA = 0.05

print("✓ Setup complete")

## 1. Load All Model Predictions

In [ ]:
# Load ground truth
y_test = pd.read_parquet(PROCESSED_DIR / "compas_y_test.parquet")['two_year_recid']

# Load all model predictions
models = ['logistic_regression', 'xgboost', 'lightgbm', 'catboost']
predictions = {}
metrics_summary = []

for model in models:
    # Load predictions
    pred_df = pd.read_parquet(PREDICTIONS_DIR / f"{model}_predictions.parquet")
    test_df = pred_df[pred_df['split'] == 'test'].reset_index(drop=True)
    
    predictions[model] = {
        'y_proba': test_df['y_proba'].values,
        'y_pred': test_df['y_pred'].values
    }
    
    # Load metrics
    with open(METRICS_DIR / f"{model}_metrics.json", 'r') as f:
        metrics = json.load(f)
    
    metrics_summary.append({
        'model': model.replace('_', ' ').title(),
        'auroc': metrics['test']['auroc'],
        'auprc': metrics['test']['auprc'],
        'accuracy': metrics['test']['accuracy'],
        'f1': metrics['test']['f1'],
        'brier_score': metrics['test']['brier_score']
    })

# Create summary DataFrame
metrics_df = pd.DataFrame(metrics_summary)

print("Loaded predictions from all models:")
print(metrics_df.to_string(index=False))
print(f"\nTest set size: {len(y_test)}")

## 2. Pairwise AUROC Comparison (DeLong Test)

DeLong's test compares two correlated ROC curves (same test set).

In [ ]:
# All pairwise comparisons
delong_results = []

for i, model1 in enumerate(models):
    for model2 in models[i+1:]:
        result = delong_test(
            y_test,
            predictions[model1]['y_proba'],
            predictions[model2]['y_proba']
        )
        
        delong_results.append({
            'model1': model1.replace('_', ' ').title(),
            'model2': model2.replace('_', ' ').title(),
            'auroc1': result['auroc1'],
            'auroc2': result['auroc2'],
            'difference': result['auroc1'] - result['auroc2'],
            'z_statistic': result['z_statistic'],
            'p_value': result['p_value']
        })

delong_df = pd.DataFrame(delong_results)

print("DeLong Test Results (Pairwise AUROC Comparisons):")
print("="*80)
print(delong_df.to_string(index=False))

# Apply multiple comparison corrections
p_values = delong_df['p_value'].values
holm_result = holm_correction(p_values, alpha=ALPHA)
bh_result = benjamini_hochberg(p_values, alpha=ALPHA)

delong_df['p_holm'] = holm_result['adjusted_p_values']
delong_df['sig_holm'] = holm_result['rejected']
delong_df['p_bh'] = bh_result['adjusted_p_values']
delong_df['sig_bh'] = bh_result['rejected']

print("\nWith Multiple Comparison Corrections:")
print(delong_df[['model1', 'model2', 'difference', 'p_value', 'p_holm', 'sig_holm']].to_string(index=False))

## 3. Pairwise Prediction Comparison (McNemar's Test)

McNemar's test compares binary predictions (correct/incorrect).

In [ ]:
# All pairwise comparisons
mcnemar_results = []

for i, model1 in enumerate(models):
    for model2 in models[i+1:]:
        result = mcnemar_test(
            y_test,
            predictions[model1]['y_pred'],
            predictions[model2]['y_pred']
        )
        
        mcnemar_results.append({
            'model1': model1.replace('_', ' ').title(),
            'model2': model2.replace('_', ' ').title(),
            'accuracy1': result['accuracy1'],
            'accuracy2': result['accuracy2'],
            'both_correct': result['contingency_table'][0][0],
            'only_1_correct': result['contingency_table'][1][0],
            'only_2_correct': result['contingency_table'][0][1],
            'both_wrong': result['contingency_table'][1][1],
            'statistic': result['statistic'],
            'p_value': result['p_value']
        })

mcnemar_df = pd.DataFrame(mcnemar_results)

print("McNemar's Test Results (Pairwise Prediction Comparisons):")
print("="*80)
print(mcnemar_df.to_string(index=False))

# Apply corrections
p_values_mc = mcnemar_df['p_value'].values
holm_mc = holm_correction(p_values_mc, alpha=ALPHA)
bh_mc = benjamini_hochberg(p_values_mc, alpha=ALPHA)

mcnemar_df['p_holm'] = holm_mc['adjusted_p_values']
mcnemar_df['sig_holm'] = holm_mc['rejected']
mcnemar_df['p_bh'] = bh_mc['adjusted_p_values']
mcnemar_df['sig_bh'] = bh_mc['rejected']

print("\nWith Multiple Comparison Corrections:")
print(mcnemar_df[['model1', 'model2', 'p_value', 'p_holm', 'sig_holm']].to_string(index=False))

## 4. Effect Sizes

Statistical significance ≠ practical significance. Calculate effect sizes.

In [ ]:
# Effect sizes for probability predictions (Cohen's d)
effect_sizes = []

for i, model1 in enumerate(models):
    for model2 in models[i+1:]:
        # Cohen's d for predicted probabilities
        d = cohens_d(
            predictions[model1]['y_proba'],
            predictions[model2]['y_proba']
        )
        
        # NNE (Number Needed to Evaluate)
        nne_result = number_needed_to_evaluate(
            y_test,
            predictions[model1]['y_pred'],
            predictions[model2]['y_pred']
        )
        
        effect_sizes.append({
            'model1': model1.replace('_', ' ').title(),
            'model2': model2.replace('_', ' ').title(),
            'cohens_d': d,
            'nne': nne_result['nne'],
            'interpretation': nne_result['interpretation']
        })

effect_df = pd.DataFrame(effect_sizes)

print("Effect Sizes:")
print("="*80)
print(effect_df.to_string(index=False))
print("\nCohen's d interpretation: <0.2=negligible, 0.2-0.5=small, 0.5-0.8=medium, >0.8=large")
print("NNE interpretation: cases needed to evaluate to get 1 additional correct prediction")

## 5. Performance Visualization

In [ ]:
# Bar plot comparing all metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = ['auroc', 'auprc', 'accuracy', 'f1']
titles = ['AUROC', 'AUPRC', 'Accuracy', 'F1 Score']

for ax, metric, title in zip(axes.flat, metrics_to_plot, titles):
    x = np.arange(len(models))
    values = [metrics_df[metrics_df['model'] == m.replace('_', ' ').title()][metric].values[0] 
              for m in models]
    
    bars = ax.bar(x, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('_', ' ').title() for m in models], rotation=45, ha='right')
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f'{title} Comparison', fontweight='bold', fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'baseline_models_comparison_bars.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved comparison bar plots")

## 6. Statistical Significance Heatmap

In [ ]:
# Create p-value matrix (DeLong test)
p_matrix = np.ones((len(models), len(models)))

for _, row in delong_df.iterrows():
    i = models.index(row['model1'].lower().replace(' ', '_'))
    j = models.index(row['model2'].lower().replace(' ', '_'))
    p_matrix[i, j] = row['p_holm']  # Use Holm-corrected p-values
    p_matrix[j, i] = row['p_holm']

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Use -log10(p) for better visualization (higher = more significant)
log_p_matrix = -np.log10(p_matrix + 1e-10)  # Add small value to avoid log(0)
np.fill_diagonal(log_p_matrix, 0)  # Set diagonal to 0

sns.heatmap(log_p_matrix, 
            xticklabels=[m.replace('_', ' ').title() for m in models],
            yticklabels=[m.replace('_', ' ').title() for m in models],
            cmap='RdYlGn', annot=True, fmt='.2f', 
            cbar_kws={'label': '-log10(p-value, Holm-corrected)'},
            ax=ax, square=True)

# Add significance threshold line
threshold = -np.log10(0.05)
ax.text(0.5, -0.15, f"Values > {threshold:.2f} indicate p < 0.05 (significant)",
        transform=ax.transAxes, ha='center', fontsize=10)

ax.set_title('Statistical Significance of AUROC Differences\n(DeLong Test with Holm Correction)',
             fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_comparison_significance_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved significance heatmap")

## 7. Summary Tables

Create publication-ready comparison tables.

In [ ]:
# Comprehensive summary table
summary_table = metrics_df.copy()
summary_table = summary_table.round(4)

# Add rankings
for metric in ['auroc', 'auprc', 'accuracy', 'f1']:
    summary_table[f'{metric}_rank'] = summary_table[metric].rank(ascending=False).astype(int)

print("\nPerformance Summary with Rankings:")
print("="*100)
print(summary_table.to_string(index=False))

# Save to CSV
summary_table.to_csv(TABLES_DIR / "baseline_model_comparison.csv", index=False)
print("\n✓ Saved: baseline_model_comparison.csv")

# Save to LaTeX
latex_table = summary_table[['model', 'auroc', 'auprc', 'accuracy', 'f1', 'brier_score']].to_latex(
    index=False,
    float_format='%.4f',
    caption='Baseline Model Performance Comparison on COMPAS Test Set',
    label='tab:baseline_comparison'
)

with open(TABLES_DIR / "baseline_model_comparison.tex", 'w') as f:
    f.write(latex_table)
print("✓ Saved: baseline_model_comparison.tex")

In [ ]:
# Statistical test summary
test_summary = delong_df[['model1', 'model2', 'difference', 'p_value', 'p_holm', 'sig_holm']].copy()
test_summary = test_summary.round(4)
test_summary.columns = ['Model 1', 'Model 2', 'AUROC Diff', 'p-value', 'p-value (Holm)', 'Significant']

print("\nStatistical Test Summary (DeLong):")
print("="*100)
print(test_summary.to_string(index=False))

# Save
test_summary.to_csv(TABLES_DIR / "model_comparison_delong_tests.csv", index=False)
print("\n✓ Saved: model_comparison_delong_tests.csv")

## 8. Save Complete Statistical Results

In [ ]:
# Compile all statistical results
statistical_results = {
    'test_set_size': int(len(y_test)),
    'alpha': ALPHA,
    'num_comparisons': len(delong_df),
    'performance_summary': metrics_df.to_dict('records'),
    'delong_tests': delong_df.to_dict('records'),
    'mcnemar_tests': mcnemar_df.to_dict('records'),
    'effect_sizes': effect_df.to_dict('records'),
    'multiple_comparison_corrections': {
        'method': 'Holm step-down',
        'num_rejected': int(holm_result['num_rejected']),
        'fwer': ALPHA
    },
    'interpretation': {
        'best_auroc': metrics_df.loc[metrics_df['auroc'].idxmax(), 'model'],
        'best_auprc': metrics_df.loc[metrics_df['auprc'].idxmax(), 'model'],
        'best_f1': metrics_df.loc[metrics_df['f1'].idxmax(), 'model'],
        'significant_differences': int(delong_df['sig_holm'].sum())
    }
}

# Save
with open(METRICS_DIR / "model_comparison_stats.json", 'w') as f:
    json.dump(statistical_results, f, indent=2)

print("✓ Saved complete statistical results")
print("\nKey Findings:")
print(f"  Best AUROC: {statistical_results['interpretation']['best_auroc']}")
print(f"  Best AUPRC: {statistical_results['interpretation']['best_auprc']}")
print(f"  Best F1: {statistical_results['interpretation']['best_f1']}")
print(f"  Significant differences (Holm-corrected): {statistical_results['interpretation']['significant_differences']} of {len(delong_df)}")

## Summary

**Statistical Model Comparison Complete:**
- ✓ DeLong tests for AUROC comparison
- ✓ McNemar's tests for prediction agreement
- ✓ Multiple comparison corrections (Holm)
- ✓ Effect sizes calculated (Cohen's d, NNE)
- ✓ Comprehensive visualizations
- ✓ Publication-ready tables (CSV, LaTeX)
- ✓ Complete statistical results exported

**Key Insights:**
- Statistical significance does not always imply practical significance
- Multiple comparison corrections are essential to control FWER
- Effect sizes provide complementary information to p-values
- Tree-based models typically outperform logistic regression on tabular data

**Next Steps:**
- 03d_hyperparameter_tuning.ipynb (Deep dive into tuning analysis)
- Phase 3: TabPFN experiments with statistical comparisons